# 05 — Modelagem dimensional e integração da Tríade

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Entrada:** tabelas tratadas de `data/processed/` (`fato_macro`, `fato_cotacoes`, `fato_contabil`).

**Modelo:** esquema estrela simplificado — uma dimensão de empresas (`dim_empresa`) ligada às tabelas de fatos, e uma tabela integrada anual (`fato_anual_integrado`) com uma linha por empresa e exercício.

**Saída:** `dim_empresa.csv`, `fato_indicadores_contabeis.csv`, `fato_macro_anual.csv`, `fato_mercado_anual.csv` e `fato_anual_integrado.csv` em `data/processed/`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 100)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"

macro = pd.read_csv(PROC_DIR / "fato_macro.csv", parse_dates=["data"])
cotacoes = pd.read_csv(PROC_DIR / "fato_cotacoes.csv", parse_dates=["data"])
contabil = pd.read_csv(PROC_DIR / "fato_contabil.csv", dtype={"cnpj": str, "cd_conta": str})

print("macro:", macro.shape, "| cotacoes:", cotacoes.shape, "| contabil:", contabil.shape)

macro: (3201, 3) | cotacoes: (11169, 9) | contabil: (9894, 8)


In [2]:
CARTEIRA = {
    "LREN3": ("Lojas Renner",  "Consumo/Varejo"),
    "ABEV3": ("Ambev",         "Consumo/Varejo"),
    "PETR4": ("Petrobras",     "Energia"),
    "EGIE3": ("Engie Brasil",  "Energia"),
    "VALE3": ("Vale",          "Commodities Exportadoras"),
    "SUZB3": ("Suzano",        "Commodities Exportadoras"),
    "WEGE3": ("WEG",           "Exportadoras Industriais"),
    "EMBR3": ("Embraer",       "Exportadoras Industriais"),
    "ITUB4": ("Itaú Unibanco", "Bancos"),
}

dim_empresa = pd.DataFrame(
    [(ticker, nome, setor) for ticker, (nome, setor) in CARTEIRA.items()],
    columns=["ticker", "empresa", "setor"],
)
dim_empresa["tipo"] = dim_empresa["setor"].map(lambda s: "Financeira" if s == "Bancos" else "Não financeira")
dim_empresa = dim_empresa.merge(contabil[["ticker", "cnpj"]].drop_duplicates(), on="ticker", how="left")

dim_empresa.to_csv(PROC_DIR / "dim_empresa.csv", index=False)
dim_empresa

,ticker,empresa,setor,tipo,cnpj
0,LREN3,Lojas Renner,Consumo/Varejo,Não financeira,92.754.738/0001-62
1,ABEV3,Ambev,Consumo/Varejo,Não financeira,07.526.557/0001-00
2,PETR4,Petrobras,Energia,Não financeira,33.000.167/0001-01
3,EGIE3,Engie Brasil,Energia,Não financeira,02.474.103/0001-19
4,VALE3,Vale,Commodities Exportadoras,Não financeira,33.592.510/0001-54
5,SUZB3,Suzano,Commodities Exportadoras,Não financeira,16.404.287/0001-55
6,WEGE3,WEG,Exportadoras Industriais,Não financeira,84.429.695/0001-11
7,EMBR3,Embraer,Exportadoras Industriais,Não financeira,07.689.002/0001-89
8,ITUB4,Itaú Unibanco,Bancos,Financeira,60.872.504/0001-23


In [3]:
CONTAS = {
    "receita_liquida": "3.01",
    "resultado_bruto": "3.03",
    "ebit": "3.05",
    "despesas_financeiras": "3.06.02",
    "lucro_liquido": "3.11",
    "ativo_total": "1",
    "ativo_circulante": "1.01",
    "caixa": "1.01.01",
    "aplicacoes_financeiras": "1.01.02",
    "passivo_circulante": "2.01",
    "emprestimos_cp": "2.01.04",
    "emprestimos_lp": "2.02.01",
    "patrimonio_liquido": "2.03",
}

nao_fin = dim_empresa.loc[dim_empresa["tipo"] == "Não financeira", "ticker"]
recorte = contabil[(contabil["ano"] == 2023)
                   & contabil["ticker"].isin(nao_fin)
                   & contabil["cd_conta"].isin(CONTAS.values())]

(recorte.groupby(["cd_conta", "ds_conta"])["ticker"]
        .apply(lambda s: ", ".join(sorted(s)))
        .reset_index(name="empresas"))

,cd_conta,ds_conta,empresas
0,1,Ativo Total,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
1,1.01,Ativo Circulante,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
2,1.01.01,Caixa e Equivalentes de Caixa,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
3,1.01.02,Aplicações Financeiras,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
4,2.01,Passivo Circulante,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
5,2.01.04,Empréstimos e Financiamentos,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
6,2.02.01,Empréstimos e Financiamentos,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
7,2.03,Patrimônio Líquido Consolidado,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
8,3.01,Receita de Venda de Bens e/ou Serviços,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
9,3.03,Resultado Bruto,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."


In [4]:
itau = contabil[(contabil["ticker"] == "ITUB4")
                & (contabil["ano"] == 2023)
                & (contabil["cd_conta"].str.count(r"\.") <= 1)]
itau[["demonstrativo", "cd_conta", "ds_conta", "valor_reais"]]

,demonstrativo,cd_conta,ds_conta,valor_reais
4195,BPA,1,Ativo Total,2.543100e+12
4196,BPA,1.01,Caixa e Equivalentes de Caixa,3.200100e+10
4197,BPA,1.02,Ativos Financeiros,2.384618e+12
4216,BPA,1.03,Tributos Diferidos,6.452100e+10
4220,BPA,1.04,Outros Ativos,2.016800e+10
4225,BPA,1.05,Investimentos,9.293000e+09
4230,BPA,1.06,Imobilizado,9.135000e+09
4233,BPA,1.07,Intangível,2.336400e+10
4236,BPP,2,Passivo Total,2.543100e+12
4237,BPP,2.01,Passivos Financeiros ao Valor Justo através do...,5.333100e+10


## Etapa 1 — Conclusão: dimensão e plano de contas

- `dim_empresa`: 9 companhias, 5 setores, classificadas em **Não financeira** (8) e **Financeira** (Itaú).
- Os 13 códigos contábeis usados nas empresas não financeiras têm **descrição única e idêntica** em todas elas (plano de contas padronizado da CVM).
- O **Itaú** segue o plano de contas de instituição financeira: Lucro Líquido = **3.09**, Patrimônio Líquido = **2.08**, Receita = **3.01** (intermediação financeira). Não há EBIT, Ativo/Passivo Circulante nem Dívida Líquida comparáveis.
- **Decisão:** para o Itaú são calculados apenas Receita, Lucro, PL e ROE; os demais indicadores ficam como não aplicáveis (N/A).

In [5]:
CONTAS_BANCO = {
    "receita_liquida": "3.01",    # Receitas da Intermediação Financeira
    "lucro_liquido": "3.09",      # Lucro/Prejuízo Consolidado do Período
    "ativo_total": "1",
    "patrimonio_liquido": "2.08",
}

(contabil[(contabil["ticker"] == "ITUB4") & contabil["cd_conta"].isin(CONTAS_BANCO.values())]
    .pivot_table(index="cd_conta", columns="ano", values="ds_conta", aggfunc="first"))

ano,2019,2020,2021,2022,2023
cd_conta,,,,,
1,Ativo Total,Ativo Total,Ativo Total,Ativo Total,Ativo Total
2.08,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado
3.01,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira
3.09,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período


In [6]:
def extrair_contas(df, mapa):
    """Seleciona as contas do mapa e organiza: uma linha por empresa/ano, uma coluna por conta."""
    codigo_para_nome = {codigo: nome for nome, codigo in mapa.items()}
    selecao = df[df["cd_conta"].isin(codigo_para_nome)].copy()
    selecao["conta"] = selecao["cd_conta"].map(codigo_para_nome)
    return (selecao.pivot_table(index=["ticker", "ano"], columns="conta",
                                values="valor_reais", aggfunc="first")
                   .reset_index())


ind_nao_fin = extrair_contas(contabil[contabil["ticker"].isin(nao_fin)], CONTAS)
ind_banco = extrair_contas(contabil[contabil["ticker"] == "ITUB4"], CONTAS_BANCO)

# Completude: contas faltantes em cada grupo (o esperado é zero)
print("Não financeiras — contas faltantes por coluna:")
print(ind_nao_fin.isna().sum())
print("\nItaú — contas faltantes por coluna:")
print(ind_banco.isna().sum())

Não financeiras — contas faltantes por coluna:
conta
ticker                    0
ano                       0
aplicacoes_financeiras    0
ativo_circulante          0
ativo_total               0
caixa                     0
despesas_financeiras      0
ebit                      0
emprestimos_cp            0
emprestimos_lp            0
lucro_liquido             0
passivo_circulante        0
patrimonio_liquido        0
receita_liquida           0
resultado_bruto           0
dtype: int64

Itaú — contas faltantes por coluna:
conta
ticker                0
ano                   0
ativo_total           0
lucro_liquido         0
patrimonio_liquido    0
receita_liquida       0
dtype: int64


In [9]:
ind = pd.concat([ind_nao_fin, ind_banco], ignore_index=True)
ind.columns.name = None  # remove o rótulo "conta" herdado do pivot

# Dívida líquida = dívida bruta (curto + longo prazo) − caixa e aplicações de curto prazo
ind["divida_bruta"] = ind["emprestimos_cp"] + ind["emprestimos_lp"]
ind["caixa_total"] = ind["caixa"] + ind["aplicacoes_financeiras"].fillna(0)
ind["divida_liquida"] = ind["divida_bruta"] - ind["caixa_total"]

# Indicadores (em fração: 0,15 = 15%)
ind["margem_bruta"] = ind["resultado_bruto"] / ind["receita_liquida"]
ind["margem_ebit"] = ind["ebit"] / ind["receita_liquida"]
ind["margem_liquida"] = ind["lucro_liquido"] / ind["receita_liquida"]
ind["roe"] = ind["lucro_liquido"] / ind["patrimonio_liquido"]
ind["liquidez_corrente"] = ind["ativo_circulante"] / ind["passivo_circulante"]
ind["divida_liquida_pl"] = ind["divida_liquida"] / ind["patrimonio_liquido"]

ind = ind.sort_values(["ticker", "ano"]).reset_index(drop=True)
ind["crescimento_receita"] = ind.groupby("ticker")["receita_liquida"].pct_change()

# Banco: margem líquida sobre receita de intermediação não é comparável às demais empresas
ind.loc[ind["ticker"] == "ITUB4", "margem_liquida"] = float("nan")

ind.to_csv(PROC_DIR / "fato_indicadores_contabeis.csv", index=False)
print("Linhas:", len(ind))

Linhas: 45


In [10]:
validacao = ind.loc[ind["ano"] == 2023,
                    ["ticker", "receita_liquida", "lucro_liquido", "roe", "margem_ebit", "liquidez_corrente"]].copy()
validacao[["receita_liquida", "lucro_liquido"]] /= 1e9  # em R$ bilhões
validacao.round(2)

,ticker,receita_liquida,lucro_liquido,roe,margem_ebit,liquidez_corrente
4,ABEV3,79.74,14.96,0.19,0.23,0.89
9,EGIE3,10.75,3.43,0.35,0.59,1.46
14,EMBR3,26.11,0.78,0.05,0.06,1.65
19,ITUB4,313.22,33.88,0.17,NaN,NaN
24,LREN3,13.65,0.98,0.10,0.07,1.63
29,PETR4,511.99,125.17,0.33,0.37,0.96
34,SUZB3,39.76,14.11,0.31,0.31,2.61
39,VALE3,208.07,40.55,0.20,0.31,1.28
44,WEGE3,32.50,5.87,0.33,0.20,1.92


## Etapa 2 — Conclusão: indicadores contábeis

- **45 registros** (9 empresas × 5 exercícios), sem contas faltantes; códigos do Itaú estáveis de 2019 a 2023.
- **Validação de ordem de grandeza (2023):** receitas e lucros compatíveis com os resultados divulgados pelas companhias (ex.: Petrobras ≈ R$ 125 bi de lucro; WEG ≈ R$ 32,5 bi de receita).
- **Indicadores em fração** (0,15 = 15%): margem bruta, margem EBIT, margem líquida, ROE, liquidez corrente, dívida líquida/PL e crescimento da receita.
- **Simplificações declaradas:** ROE sobre o PL do fim do exercício; alavancagem medida por Dívida Líquida/PL (o EBITDA exigiria a depreciação, ausente da DRE); lucro consolidado (inclui minoritários).
- **Itaú:** margens, liquidez e dívida líquida não aplicáveis (N/A).

In [11]:
macro["ano"] = macro["data"].dt.year
series = macro.pivot_table(index=["data", "ano"], columns="serie", values="valor").reset_index()

selic = series.dropna(subset=["selic_meta"]).groupby("ano")["selic_meta"].agg(["mean", "last"])
ptax = series.dropna(subset=["ptax_venda"]).groupby("ano")["ptax_venda"].agg(["mean", "first", "last"])
ipca_mensal = series.dropna(subset=["ipca_mensal"])
ipca_ano = ipca_mensal.groupby("ano")["ipca_mensal"].apply(lambda s: ((1 + s / 100).prod() - 1) * 100)
ipca_12m_dez = series.dropna(subset=["ipca_12m"]).groupby("ano")["ipca_12m"].last()

macro_anual = pd.DataFrame({
    "selic_media_pct": selic["mean"],
    "selic_fim_ano_pct": selic["last"],
    "ipca_ano_pct": ipca_ano,
    "ipca_12m_dez_pct": ipca_12m_dez,          # série oficial 13522, para validação
    "ptax_media": ptax["mean"],
    "ptax_fim_ano": ptax["last"],
    "ptax_variacao_ano_pct": (ptax["last"] / ptax["first"] - 1) * 100,
}).reset_index()

# Juro real ex-post pela equação de Fisher: (1 + Selic) / (1 + IPCA) − 1
macro_anual["juro_real_pct"] = ((1 + macro_anual["selic_media_pct"] / 100)
                                / (1 + macro_anual["ipca_ano_pct"] / 100) - 1) * 100

macro_anual.to_csv(PROC_DIR / "fato_macro_anual.csv", index=False)
macro_anual.round(2)

,ano,selic_media_pct,selic_fim_ano_pct,ipca_ano_pct,ipca_12m_dez_pct,ptax_media,ptax_fim_ano,ptax_variacao_ano_pct,juro_real_pct
0,2019,6.04,4.50,4.31,4.31,3.95,4.03,4.44,1.66
1,2020,2.89,2.00,4.52,4.52,5.16,5.20,29.23,-1.56
2,2021,4.54,9.25,10.06,10.06,5.40,5.58,8.09,-5.02
3,2022,12.55,13.75,5.78,5.79,5.17,5.22,-7.34,6.39
4,2023,13.30,11.75,4.62,4.62,5.00,4.84,-9.40,8.29


In [12]:
series["mes"] = series["data"].dt.to_period("M").dt.to_timestamp()
macro_mensal = series.groupby("mes").agg(
    selic_meta_pct=("selic_meta", "last"),
    ipca_mensal_pct=("ipca_mensal", "last"),
    ipca_12m_pct=("ipca_12m", "last"),
    ptax_media=("ptax_venda", "mean"),
).reset_index()

macro_mensal.to_csv(PROC_DIR / "fato_macro_mensal.csv", index=False)
print("Meses:", len(macro_mensal))
macro_mensal.tail()

Meses: 60


,mes,selic_meta_pct,ipca_mensal_pct,ipca_12m_pct,ptax_media
55,2023-08-01,13.25,0.23,4.61,4.903543
56,2023-09-01,12.75,0.26,5.19,4.936990
57,2023-10-01,12.75,0.24,4.82,5.064843
58,2023-11-01,12.25,0.28,4.68,4.898340
59,2023-12-01,11.75,0.56,4.62,4.897245


In [13]:
cot = cotacoes.sort_values(["ticker", "data"]).copy()
cot["ano"] = cot["data"].dt.year
cot["retorno_diario"] = cot.groupby("ticker")["fechamento_ajustado"].pct_change()

mercado = cot.groupby(["ticker", "ano"]).agg(
    preco_inicio=("fechamento_ajustado", "first"),
    preco_fim=("fechamento_ajustado", "last"),
    vol_diaria=("retorno_diario", "std"),
    volume_financeiro_medio=("volume_financeiro", "mean"),
    pregoes=("data", "count"),
).reset_index()

# Retorno anual: fim do ano ÷ fim do ano anterior − 1 (em 2019, base = 1º pregão do ano)
base = mercado.groupby("ticker")["preco_fim"].shift(1).fillna(mercado["preco_inicio"])
mercado["retorno_anual"] = mercado["preco_fim"] / base - 1
mercado["volatilidade_anual"] = mercado["vol_diaria"] * 252 ** 0.5
mercado = mercado.drop(columns=["preco_inicio", "vol_diaria"])

mercado.to_csv(PROC_DIR / "fato_mercado_anual.csv", index=False)

# Retorno anual por ativo (%)
(mercado.pivot_table(index="ticker", columns="ano", values="retorno_anual") * 100).round(1)

ano,2019,2020,2021,2022,2023
ticker,,,,,
ABEV3,18.6,-14.0,2.9,-0.9,-0.5
EGIE3,54.1,-10.2,-8.5,6.0,28.1
EMBR3,-8.4,-55.1,180.5,-42.3,56.5
ITUB4,8.3,-11.2,-31.4,24.2,42.3
LREN3,43.7,-21.9,-36.9,-13.8,-11.2
PETR4,29.6,-6.1,23.5,37.9,96.8
SUZB3,3.9,47.5,2.7,-14.9,18.0
VALE3,7.1,70.9,4.9,25.0,-5.7
WEGE3,95.8,120.3,-11.9,18.9,-2.6


## Etapa 3 — Conclusão: macroeconomia e mercado anuais

- **Validação do IPCA:** o acumulado anual calculado pela composição dos índices mensais coincide com a série oficial IPCA 12m de dezembro (4,31 / 4,52 / 10,06 / 5,78 / 4,62%).
- **Ciclo monetário capturado:** Selic de 2,00% (fim de 2020) a 13,75% (fim de 2022); juro real ex-post (Fisher) de −5,0% em 2021 a +8,3% em 2023.
- **Choque cambial:** PTAX +29,2% em 2020.
- **Unidades:** dados macro em pontos percentuais (sufixo `_pct`); retornos e volatilidades em fração.
- **Limitações:** retorno de 2019 com base no 1º pregão do ano (sem dados de 2018); o retorno do ITUB4 em 2021 pode estar subestimado pela cisão da participação na XP Inc., possivelmente não refletida no preço ajustado.

In [14]:
integrado = (
    ind.merge(mercado, on=["ticker", "ano"], how="left", validate="one_to_one")
       .merge(macro_anual, on="ano", how="left", validate="many_to_one")
       .merge(dim_empresa[["ticker", "empresa", "setor", "tipo"]], on="ticker",
              how="left", validate="many_to_one")
)

# Organiza as colunas: identificação → contábil → mercado → macro
identificacao = ["ticker", "empresa", "setor", "tipo", "ano"]
contabeis = [c for c in ind.columns if c not in ("ticker", "ano")]
de_mercado = [c for c in mercado.columns if c not in ("ticker", "ano")]
macroeconomicos = [c for c in macro_anual.columns if c != "ano"]
integrado = integrado[identificacao + contabeis + de_mercado + macroeconomicos]

integrado.to_csv(PROC_DIR / "fato_anual_integrado.csv", index=False)
print("Formato:", integrado.shape)

Formato: (45, 41)


In [15]:
nulos = integrado.isna().sum()
nulos[nulos > 0]

aplicacoes_financeiras    5
ativo_circulante          5
caixa                     5
despesas_financeiras      5
ebit                      5
emprestimos_cp            5
emprestimos_lp            5
passivo_circulante        5
resultado_bruto           5
divida_bruta              5
caixa_total               5
divida_liquida            5
margem_bruta              5
margem_ebit               5
margem_liquida            5
liquidez_corrente         5
divida_liquida_pl         5
crescimento_receita       9
dtype: int64

In [16]:
previa = integrado.groupby(["setor", "ano"])["retorno_anual"].mean().unstack() * 100
previa.loc["Selic média (%)"] = macro_anual.set_index("ano")["selic_media_pct"]
previa.round(1)

ano,2019,2020,2021,2022,2023
setor,,,,,
Bancos,8.3,-11.2,-31.4,24.2,42.3
Commodities Exportadoras,5.5,59.2,3.8,5.0,6.1
Consumo/Varejo,31.2,-17.9,-17.0,-7.3,-5.9
Energia,41.9,-8.1,7.5,22.0,62.5
Exportadoras Industriais,43.7,32.6,84.3,-11.7,26.9
Selic média (%),6.0,2.9,4.5,12.5,13.3


## Etapa 4 — Conclusão: tabela integrada da Tríade

- **`fato_anual_integrado`: 45 registros (9 empresas × 5 exercícios) × 41 colunas**, unindo indicadores contábeis (CVM), de mercado (B3) e macroeconômicos (BACEN), além da dimensão de empresas.
- **Integridade dos JOINs** garantida pelo parâmetro `validate` (`one_to_one` e `many_to_one`): nenhuma linha duplicada.
- **Nulos esperados e justificados:** 5 por coluna nos indicadores não aplicáveis ao Itaú (instituição financeira) e 9 no crescimento da receita (2019, sem exercício anterior).
- **Modelo:** constelação de fatos (variação do esquema estrela), com `dim_empresa` compartilhada; a dimensão tempo é representada pelas colunas `data`/`ano`.

**Primeiras evidências (associação, não causalidade):**
- Consumo/Varejo teve retorno médio negativo de 2020 a 2023, período de alta da Selic.
- Energia (+62,5%) e Bancos (+42,3%) tiveram os maiores retornos em 2023, com a Selic média em 13,3%.
- Commodities Exportadoras (+59,2%) destacaram-se em 2020, ano de alta de 29% do dólar.